<a href="https://colab.research.google.com/github/AyushiB56/Neural-Network/blob/main/vanilla_RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [189]:
import torch
import os
import glob
import string
import unicodedata
import random
import io

In [190]:

every_ascii_character= string.ascii_letters + " .,:;"
len_of_char= len(every_ascii_character)

In [191]:


def unicode_to_ascii(s):
  return ''. join(c for c in unicodedata.normalize('NFD',s) if  unicodedata.category(c) != 'Mn' and c in every_ascii_character)

In [192]:
def load_data():
  countries=[]
  country_people_name={}
  def find_files(path):
    return glob.glob(path)
  def read_lines(filename):
    lines= io.open(filename,encoding='utf-8').read().strip().split('\n')

    return [unicode_to_ascii(line) for line in lines]

  for files in find_files('sample_data/data/*.txt'):
    country =  os.path.splitext(os.path.basename(files))[0]

    countries.append(country)

    lines= read_lines(files)
    country_people_name[country]=lines
  return country_people_name,countries





In [193]:
import numpy as np

In [194]:
def find_letter(letter):
  return every_ascii_character.find(letter)

In [195]:
def letter_to_ohe(letter):
  letter_to_tensor= torch.zeros(1,len_of_char)
  letter_to_tensor[0][find_letter(letter)]=1
  return letter_to_tensor

In [196]:
def line_to_ohe(line):
  tensor = torch.zeros(len(line), 1, len_of_char)
  for i, letter in enumerate(line):
        tensor[i][0][find_letter(letter)] = 1
  return tensor




In [197]:
def random_set():
  def random_choice(a):
    return a[random.randint(0,len(countries)-1)]
  country_people_name,countries= load_data()
  country= random_choice(countries)
  country_tensor= torch.tensor([countries.index(country)],dtype=torch.long)

  people_name= random_choice(country_people_name[country_name])
  line_tensor= line_to_ohe(people_name)
  return country_tensor,country,people_name,line_tensor

In [198]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt




In [199]:
class Rnn_module(nn.Module):
  def __init__(self, input_size, hidden_size, output_size):

    super(Rnn_module, self).__init__()
    self.hidden_size= hidden_size
    self.i2h=nn.Linear(input_size + hidden_size, hidden_size)
    self.i2o=nn.Linear(input_size + hidden_size, output_size)
    self.softmax= nn.LogSoftmax(dim=1)
  def forward(self, input_tensor, hidden_tensor):
    combined= torch.cat((input_tensor, hidden_tensor),1)
    hidden = self.i2h(combined)
    output = self.i2o(combined)
    output = self.softmax(output)


    return output, hidden
  def init_hidden(self):
    return torch.zeros(1, self.hidden_size)


In [208]:
country_tensor,country,people_name,line_tensor= random_set()
hidden_size=128
rnn= Rnn_module(len_of_char,hidden_size,len(countries))
optimizer= torch.optim.Adam(rnn.parameters(), lr=0.005)
loss= nn.NLLLoss()
def train():

  hidden= rnn.init_hidden()


  for i in range(len(line_tensor)-1):
      output, hidden = rnn(line_tensor[i], hidden.detach())

  loss_value= loss(output,country_tensor)
  optimizer.zero_grad()
  loss_value.backward()
  optimizer.step()


    #print("loss per epoch", loss_value.item())
  return output,country,people_name,loss_value.item()



In [201]:
def category_output(output):
  max_out= torch.argmax(output).item()
  return countries[max_out]

In [202]:
loss_epoch= []

In [ ]:
all_loss=[]
current_loss=0
for i in range(100000):
  output,country,people_name,train_loss=train()
  current_loss+=train_loss

  if (i+1)%1000==0:
    all_loss.append(current_loss/1000)
    current_loss=0

In [ ]:
plt.figure()
plt.plot(all_loss)
plt.show()

In [181]:

def predict(input_line):
    print(f"\n> {input_line}")
    with torch.no_grad():
        line_tensor = line_to_ohe(input_line)

        hidden = rnn.init_hidden()

        for i in range(line_tensor.size()[0]):
            output, hidden = rnn(line_tensor[i], hidden)

        guess = category_output(output)
        print(guess)


In [182]:
predict('Dubicki')


> Dubicki
Greek


In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from utils import ALL_LETTERS, N_LETTERS
from utils import load_data, letter_to_tensor, line_to_tensor, random_training_example


class RNN(nn.Module):
    # implement RNN from scratch rather than using nn.RNN
    def __init__(self, input_size, hidden_size, output_size):
        super(RNN, self).__init__()

        self.hidden_size = hidden_size
        self.i2h = nn.Linear(input_size + hidden_size, hidden_size)
        self.i2o = nn.Linear(input_size + hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, input_tensor, hidden_tensor):
        combined = torch.cat((input_tensor, hidden_tensor), 1)

        hidden = self.i2h(combined)
        output = self.i2o(combined)
        output = self.softmax(output)
        return output, hidden

    def init_hidden(self):
        return torch.zeros(1, self.hidden_size)

category_lines, all_categories = load_data()
n_categories = len(all_categories)

n_hidden = 128
rnn = RNN(N_LETTERS, n_hidden, n_categories)

# one step
input_tensor = letter_to_tensor('A')
hidden_tensor = rnn.init_hidden()

output, next_hidden = rnn(input_tensor, hidden_tensor)
#print(output.size())
#print(next_hidden.size())

# whole sequence/name
input_tensor = line_to_tensor('Albert')
hidden_tensor = rnn.init_hidden()

output, next_hidden = rnn(input_tensor[0], hidden_tensor)
#print(output.size())
#print(next_hidden.size())

#
def category_from_output(output):
    category_idx = torch.argmax(output).item()
    return all_categories[category_idx]

print(category_from_output(output))

criterion = nn.NLLLoss()
learning_rate = 0.005
optimizer = torch.optim.SGD(rnn.parameters(), lr=learning_rate)

def train(line_tensor, category_tensor):
    hidden = rnn.init_hidden()

    for i in range(line_tensor.size()[0]):
        output, hidden = rnn(line_tensor[i], hidden)

    loss = criterion(output, category_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return output, loss.item()

current_loss = 0
all_losses = []
plot_steps, print_steps = 1000, 5000
n_iters = 100000
for i in range(n_iters):
    category, line, category_tensor, line_tensor = random_training_example(category_lines, all_categories)

    output, loss = train(line_tensor, category_tensor)
    current_loss += loss

    if (i+1) % plot_steps == 0:
        all_losses.append(current_loss / plot_steps)
        current_loss = 0

    if (i+1) % print_steps == 0:
        guess = category_from_output(output)
        correct = "CORRECT" if guess == category else f"WRONG ({category})"
        print(f"{i+1} {(i+1)/n_iters*100} {loss:.4f} {line} / {guess} {correct}")


plt.figure()
plt.plot(all_losses)
plt.show()

def predict(input_line):
    print(f"\n> {input_line}")
    with torch.no_grad():
        line_tensor = line_to_tensor(input_line)

        hidden = rnn.init_hidden()

        for i in range(line_tensor.size()[0]):
            output, hidden = rnn(line_tensor[i], hidden)

        guess = category_from_output(output)
        print(guess)


while True:
    sentence = input("Input:")
    if sentence == "quit":
        break

    predict(sentence)


ModuleNotFoundError: No module named 'utils'